# Explainable AI — XGBoost

This notebook explains the validation-selected XGBoost model with SHAP. `V1`–`V28` remain anonymized PCA components; no semantic meaning is invented for them.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_dataset, split_dataset
from src.explainability import build_shap_explainer, compute_shap_values
from src.models import build_xgboost
from src.preprocessing import build_tree_preprocessor, fit_transform_splits

random_state = 42
rng = np.random.default_rng(random_state)


In [ ]:
transaction_dataset = load_dataset(project_root / "data" / "creditcard.csv")
(
    training_features,
    validation_features,
    test_features,
    training_labels,
    validation_labels,
    test_labels,
) = split_dataset(transaction_dataset, random_state=random_state)

tree_preprocessor = build_tree_preprocessor(training_features.columns.tolist())
tree_training_features, tree_validation_features, _, _ = fit_transform_splits(
    tree_preprocessor,
    training_features,
    validation_features,
    test_features,
)

fraud_count = int(training_labels.sum())
legitimate_count = len(training_labels) - fraud_count

xgboost_model = build_xgboost(random_state=random_state)
xgboost_model.set_params(scale_pos_weight=legitimate_count / fraud_count)
xgboost_model.fit(tree_training_features, training_labels)


In [ ]:
background_size = min(1000, len(tree_training_features))
explanation_size = min(500, len(tree_validation_features))

background_indices = rng.choice(len(tree_training_features), size=background_size, replace=False)
explanation_indices = rng.choice(len(tree_validation_features), size=explanation_size, replace=False)

background_features = tree_training_features[background_indices]
explained_features = tree_validation_features[explanation_indices]

explainer = build_shap_explainer(xgboost_model, background_features)
shap_values = compute_shap_values(explainer, explained_features)

print("SHAP matrix:", shap_values.shape)


In [ ]:
figure_directory = project_root / "outputs" / "figures"
figure_directory.mkdir(parents=True, exist_ok=True)

feature_names = training_features.columns.tolist()

shap.summary_plot(
    shap_values,
    explained_features,
    feature_names=feature_names,
    show=False,
)
plt.tight_layout()
plt.savefig(figure_directory / "xgboost_shap_summary.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
mean_absolute_shap = np.abs(shap_values).mean(axis=0)
importance_table = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_absolute_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

importance_table.head(10)


In [ ]:
local_index = 0
local_contributions = (
    pd.DataFrame({
        "feature": feature_names,
        "shap_value": shap_values[local_index],
    })
    .assign(abs_shap=lambda table: table["shap_value"].abs())
    .sort_values("abs_shap", ascending=False)
    .head(10)
    .sort_values("shap_value")
)

plt.figure()
plt.barh(local_contributions["feature"], local_contributions["shap_value"])
plt.xlabel("SHAP contribution")
plt.title("Local explanation — one validation transaction")
plt.tight_layout()
plt.savefig(figure_directory / "xgboost_shap_local_example.png", dpi=160, bbox_inches="tight")
plt.show()


## Interpretation

SHAP identifies which anonymized components contribute most to the model output, globally and for an individual validation transaction. Because `V1`–`V28` are PCA-anonymized, the analysis reports their influence without assigning real-world meanings to those components.
